# Run GLiClass 9B and 27B on a RunPod GPU pod

The 9B and 27B models need a GPU. This notebook runs them on a single-GPU RunPod pod and scores the results.
It runs the same job as `uv run run_on_pod.py` from a terminal:

1. Create or reuse the pod defined in `POD`.
2. Upload `src/` and `questions.json`, and install the local versions of `gliclass` and `pandas`.
3. Run every question seven ways for each model: `label_graph`, `label_nograph`, `description_graph`, `label_noedges`, `label_norules`, `label_nograph_norules` and `description_nograph_norules`.
4. Download `results/`.
5. End the pod: `terminate` deletes it and its volume; `stop` keeps the volume.

The pod is billed while it exists. It is ended however the job finishes, including when you interrupt the kernel.

`.env`, next to this notebook, needs:
- `RUNPOD_API_KEY`
- `RUNPOD_SSH_KEY`: the private key whose public key is registered with RunPod (default `~/.ssh/id_ed25519`)
- `HF_TOKEN` (optional), for Hugging Face downloads on the pod

## 1. Settings

`POD` and `MODELS` are defined in `run_on_pod.py`. The pod table leaves out the pod's environment, which holds the Hugging Face token.

**GPU availability.** RunPod's free GPUs change from minute to minute. If none of the chosen type is free, the job stops at once with
`There are no longer any instances available`; no pod is created, so nothing is billed.
Set `GPU_TYPE` and `CLOUD_TYPE` below to another GPU with at least 80 GB (27B in bfloat16 needs about 54 GB) and rerun section 2.
The community cloud is cheaper and often has more free GPUs. Suitable types, cheapest first:

| `GPU_TYPE` | `CLOUD_TYPE` |
|---|---|
| `NVIDIA A100 80GB PCIe` | `COMMUNITY` or `SECURE` |
| `NVIDIA A100-SXM4-80GB` | `COMMUNITY` or `SECURE` |
| `NVIDIA RTX PRO 6000 Blackwell Server Edition` | `COMMUNITY` or `SECURE` |
| `NVIDIA H100 PCIe` | `SECURE` |

`runpod.get_gpus()` lists every GPU type and its id.

In [1]:
import pandas as pd
from IPython.display import display

import run_on_pod

display(pd.Series({key: value for key, value in run_on_pod.POD.items() if key != "env"}, name="pod").to_frame())
display(pd.DataFrame(run_on_pod.MODELS).set_index("model_name"))
print("Packages installed on the pod:", run_on_pod.POD_PACKAGES)

,pod
name,glidecide
image,runpod/pytorch:1.0.2-cu1281-torch280-ubuntu2404
gpu_type,NVIDIA H100 80GB HBM3
cloud_type,SECURE
container_disk_gb,50
volume_gb,150
ports,22/tcp


,model_id,device,dtype,max_length
model_name,,,,
9b,knowledgator/gliclass-v4-9b,cuda:0,bfloat16,8192
27b,knowledgator/gliclass-v4-27b,cuda:0,bfloat16,8192


Packages installed on the pod: gliclass==0.1.21 pandas==3.1.0


In [2]:
MODEL_NAMES = ["9b", "27b"]
GPU_TYPE = run_on_pod.POD["gpu_type"]      # or another type from the table above
CLOUD_TYPE = run_on_pod.POD["cloud_type"]  # "SECURE" or "COMMUNITY"
END = "terminate"  # or "stop" to keep the volume, with the downloaded models, for the next run
FRESH = False      # True deletes any existing pod first             

## 2. Run the job

Progress is printed as the job runs. The first run downloads the models into the pod's Hugging Face cache (9B about 18 GB, 27B about 54 GB).
If the pod cannot be created, see **GPU availability** in section 1.

In [3]:
run_on_pod.POD.update(gpu_type=GPU_TYPE, cloud_type=CLOUD_TYPE)
succeeded = run_on_pod.run_job(MODEL_NAMES, end=END, fresh=FRESH)
print("All models ran." if succeeded else "Some models did not run; see the log above.")

[14:05:43] Plan: 9b, 27b on 1 x NVIDIA H100 80GB HBM3; the pod will be terminated at the end
[14:05:43] Creating pod 'glidecide': 1 x NVIDIA H100 80GB HBM3, SECURE cloud
[14:05:45] Pod b6dhxpi7q4540s is RUNNING after 1 s
[14:05:46] Waiting for RunPod to report the SSH address (0 s)
[14:05:56] Waiting for RunPod to report the SSH address (11 s)
[14:06:07] Waiting for RunPod to report the SSH address (21 s)
[14:06:17] Connecting over SSH to 64.247.201.55:15445
[14:06:18] Uploading src/ and questions.json
[14:06:19] Unpacking into a clean /workspace/glidecide and installing gliclass==0.1.21 pandas==3.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 19.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 28.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 11.6 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 9.0 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 1.5 MB/s  0:00:00
   ━━━━━━━━━━

[21:08:05] Still loading (30 s)
[21:08:35] Still loading (60 s)
[21:09:05] Still loading (90 s)
[21:09:35] Still loading (120 s)
[21:10:05] Still loading (150 s)


Loading weights: 100%|██████████| 824/824 [00:00<00:00, 10796.28it/s]
[transformers] You are using a model of type `GLiClass` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


[21:10:32] Loaded in 177 s
[21:10:32] Run 1/7: 9b_label_graph


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


  Q01  FLAG  0.538  0.66 s
  Q02  FAMILY_HISTORY  0.995  0.07 s
  Q03  BOTH  0.937  0.08 s
  Q04  HISTORY_ONLY  0.653  0.06 s
  Q05  BEFORE  0.980  0.05 s
  Q06  INSUFFICIENT_INFORMATION  0.716  0.06 s
  Q07  DOSE_30  0.910  0.05 s
  Q08  UNIT_ML  0.978  0.06 s
  Q09  AFTER  0.832  0.06 s
  Q10  NO  0.520  0.07 s
  Q11  INSUFFICIENT_INFORMATION  0.392  0.06 s
  Q12  INSUFFICIENT_INFORMATION  0.403  0.07 s
  Q13  YES  0.467  0.05 s
  Q14  NO  0.643  0.06 s
  Q15  INFLUENZA_ONLY  0.752  0.06 s
  Q16  PARTIAL_DOSE  0.534  0.08 s
  Q17  DIFFERENT_CODES  0.872  0.06 s
  Q18  MATCH  0.944  0.06 s
  Q19  ONE  0.818  0.07 s
  Q20  EVERY_ADMISSION  0.504  0.06 s
  Q21  SAME  0.650  0.09 s
  Q22  YES  0.754  0.06 s
  Q23  INSUFFICIENT_INFORMATION  0.764  0.05 s
  Q24  INSUFFICIENT_INFORMATION  0.783  0.06 s
  Q25  INSUFFICIENT_INFORMATION  0.709  0.06 s
  Q26  INSUFFICIENT_INFORMATION  0.831  0.06 s
  Q27  INSUFFICIENT_INFORMATION  0.389  0.07 s
  Q28  MATCH  0.924  0.06 s
  Q29  MATCH  0.936  0

... (4 outputs left out)

Loading weights: 100%|██████████| 1248/1248 [00:00<00:00, 2116.99it/s]
[transformers] You are using a model of type `GLiClass` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


[21:16:59] Still loading (360 s)
[21:17:06] Loaded in 366 s
[21:17:06] Run 1/7: 27b_label_graph


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


  Q01  DO_NOT_FLAG  0.480  0.56 s
  Q02  FAMILY_HISTORY  0.990  0.12 s
  Q03  BOTH  0.980  0.15 s
  Q04  HISTORY_ONLY  0.771  0.11 s
  Q05  BEFORE  0.995  0.10 s
  Q06  INSUFFICIENT_INFORMATION  0.922  0.13 s
  Q07  DOSE_30  0.960  0.11 s
  Q08  UNIT_ML  0.986  0.10 s
  Q09  INSUFFICIENT_INFORMATION  0.908  0.11 s
  Q10  NO  0.453  0.13 s
  Q11  NOT_GIVEN  0.807  0.11 s
  Q12  OUTSIDE_WINDOW  0.976  0.13 s
  Q13  NO  0.489  0.10 s
  Q14  NO  0.791  0.13 s
  Q15  INFLUENZA_ONLY  0.861  0.11 s
  Q16  INSUFFICIENT_INFORMATION  0.499  0.15 s
  Q17  DIFFERENT_CODES  0.998  0.13 s
  Q18  MATCH  0.966  0.12 s
  Q19  ONE  0.965  0.16 s
  Q20  SOME_ADMISSIONS  0.717  0.13 s
  Q21  DIFFERENT  0.894  0.23 s
  Q22  YES  0.901  0.10 s
  Q23  INSUFFICIENT_INFORMATION  0.969  0.10 s
  Q24  INSUFFICIENT_INFORMATION  0.832  0.10 s
  Q25  INSUFFICIENT_INFORMATION  0.906  0.12 s
  Q26  INSUFFICIENT_INFORMATION  0.474  0.11 s
  Q27  PARTIAL_DOSE  0.606  0.14 s
  Q28  MATCH  0.938  0.11 s
  Q29  MATCH  0.9

## 3. Score the results

Reads the downloaded `results/<model>_<run>.json` files: each holds the run's settings, then one answer per question. 

In [4]:
from pathlib import Path

from src.evaluate import accuracy, load_results, score
from src.questions import RUNS, load_questions

questions = load_questions("questions.json")
scored = {}
for model in MODEL_NAMES:
    for run in RUNS:
        path = Path(f"results/{model}_{run}.json")
        if path.exists():
            settings, answers = load_results(path)
            scored[(model, run)] = score(answers, questions, settings["include_graph"])
        else:
            print(f"Missing: {path}")

display(pd.concat({key: accuracy(table) for key, table in scored.items()}, names=["model", "run"]).droplevel(-1))

correct  total  accuracy
model run                                                  
9b    label_graph                       36     48  0.750000
      label_nograph                      8     48  0.166667
      description_graph                 38     48  0.791667
      label_noedges                     35     48  0.729167
      label_norules                     36     48  0.750000
      label_nograph_norules              0     48  0.000000
      description_nograph_norules        1     48  0.020833
27b   label_graph                       38     48  0.791667
      label_nograph                      7     48  0.145833
      description_graph                 40     48  0.833333
      label_noedges                     37     48  0.770833
      label_norules                     35     48  0.729167
      label_nograph_norules              0     48  0.000000
      description_nograph_norules        6     48  0.125000

In [5]:
for model in MODEL_NAMES:
    if (model, "label_graph") in scored:
        print(f"{model}, label_graph")
        display(accuracy(scored[(model, "label_graph")], by="question_type"))

9b, label_graph


,question_type,correct,total,accuracy
0,computation,11,16,0.687500
1,graph,6,13,0.461538
2,lookup,5,5,1.000000
3,semantic,14,14,1.000000


27b, label_graph


,question_type,correct,total,accuracy
0,computation,10,16,0.625000
1,graph,9,13,0.692308
2,lookup,5,5,1.000000
3,semantic,14,14,1.000000
